# SupportPilot: fine-tune DistilBERT ticket classifiers (Colab GPU)

**Generated by `scripts/build_colab_notebook.py`. Don't edit by hand.**

Trains two classifiers on the same deduplicated split as the TF-IDF baseline:
- `queue`: 10 routing categories
- `priority`: low / medium / high

**Run it from VS Code:** *Select Kernel → Colab → New Colab Server → GPU (T4)*, sign in with Google,
then *Run All*. Outputs are saved to Google Drive in `MyDrive/SupportPilot/ticket_classifier/<RUN_NAME>/`.
Download that folder into the project as `models/distilbert/` and run
`python scripts/import_colab_results.py`.

## 1. Check the GPU

In [1]:
import torch
assert torch.cuda.is_available(), "No GPU: choose a GPU runtime (T4) when creating the Colab server"
print(torch.cuda.get_device_name(0))

Tesla T4


## 2. Install libraries

In [2]:
%pip install -q -U "transformers>=5,<6" accelerate datasets scikit-learn  # same major version as local

## 3. Data: same code as `src/supportpilot/classifier/data.py`

In [3]:
"""Ticket dataset: load, clean, deduplicate, split. Single source of truth for the local baseline
AND the Colab notebook (scripts/build_colab_notebook.py embeds this file), so both are trained and
tested on exactly the same rows. `fingerprint()` lets the notebook verify that.

Dataset: Tobi-Bueck/customer-support-tickets (Hugging Face, CC BY-NC 4.0), English rows only.
Labels: `queue` (10 routing categories) and `priority` (low / medium / high).

Leakage guard: 16% of tickets are exact copies of another ticket. Duplicates are removed by
normalized body BEFORE splitting, so no test ticket has a copy in training.
"""

import hashlib
import re

import pandas as pd
from sklearn.model_selection import train_test_split

DATASET = "Tobi-Bueck/customer-support-tickets"
SEED = 42
TASKS = ("queue", "priority")


def _norm(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip().lower()


def load_tickets() -> pd.DataFrame:
    from datasets import load_dataset

    raw = load_dataset(DATASET, split="train").to_pandas()
    df = raw[raw["language"] == "en"].copy()
    df["ticket_id"] = df.index.astype(int)  # row index in the source dataset, stable across runs
    df["subject"] = df["subject"].fillna("").str.strip()
    df["body"] = df["body"].fillna("").str.strip()
    df = df[df["body"].str.len() > 0]
    df["text"] = (df["subject"] + "\n\n" + df["body"]).str.strip()
    df = df[~df["body"].map(_norm).duplicated(keep="first")]
    return df[["ticket_id", "text", "queue", "priority"]].reset_index(drop=True)


def split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """80 / 10 / 10, stratified by queue (the more imbalanced label)."""
    train, rest = train_test_split(df, test_size=0.2, stratify=df["queue"], random_state=SEED)
    val, test = train_test_split(rest, test_size=0.5, stratify=rest["queue"], random_state=SEED)
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)


def fingerprint(df: pd.DataFrame) -> str:
    """Short hash of the ticket ids in a split, to check two environments use identical splits."""
    ids = ",".join(map(str, sorted(df["ticket_id"])))
    return hashlib.sha256(ids.encode()).hexdigest()[:12]

In [4]:
EXPECTED = {"train": "74febd41127e", "val": "f0e9843bfc51", "test": "ca6c49d5b3dc"}

df = load_tickets()
train_df, val_df, test_df = split(df)
got = {"train": fingerprint(train_df), "val": fingerprint(val_df), "test": fingerprint(test_df)}
print(len(df), "tickets |", {k: len(v) for k, v in (("train", train_df), ("val", val_df), ("test", test_df))})
assert got == EXPECTED, f"Split differs from the local baseline: {got} != {EXPECTED}"
print("Split fingerprints match the local baseline:", got)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


23654 tickets | {'train': 18923, 'val': 2365, 'test': 2366}
Split fingerprints match the local baseline: {'train': '74febd41127e', 'val': 'f0e9843bfc51', 'test': 'ca6c49d5b3dc'}


## 4. Output folder on Google Drive

In [5]:
import os
RUN_NAME = "run2"  # each run gets its own folder, earlier runs are kept
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = f"/content/drive/MyDrive/SupportPilot/ticket_classifier/{RUN_NAME}"
except Exception as e:  # Drive not available: keep results on the runtime disk
    print("Drive mount failed:", e)
    OUT_DIR = f"/content/ticket_classifier/{RUN_NAME}"
os.makedirs(OUT_DIR, exist_ok=True)
print("Saving to", OUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving to /content/drive/MyDrive/SupportPilot/ticket_classifier/run2


## 5. Fine-tune

In [6]:
import json, time
import numpy as np
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256
# run1: 4 epochs, unweighted loss -> underfit (val macro-F1 still rising at the last epoch) and weak on rare classes
HPARAMS = {"learning_rate": 5e-5, "batch_size": 32, "epochs": 10, "early_stopping_patience": 2,
           "warmup_fraction": 0.1, "weight_decay": 0.01, "class_weights": "balanced"}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
lengths = [len(t) for t in tokenizer(df["text"].tolist()[:3000])["input_ids"]]
print("token length p50/p95/max:", int(np.percentile(lengths, 50)), int(np.percentile(lengths, 95)), max(lengths))


def to_ds(frame, task, label2id):
    ds = Dataset.from_dict({"text": frame["text"].tolist(), "label": [label2id[x] for x in frame[task]]})
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=["text"])


class WeightedTrainer(Trainer):
    # cross-entropy with per-class weights, like class_weight="balanced" in the TF-IDF baseline

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits.float()  # fp16 logits -> fp32 for a stable weighted loss
        loss = torch.nn.functional.cross_entropy(logits, labels, weight=self.class_weights.to(logits.device))
        return (loss, outputs) if return_outputs else loss


def balanced_weights(series, labels):
    counts = series.value_counts()
    return torch.tensor([len(series) / (len(labels) * counts[l]) for l in labels], dtype=torch.float)


def train_task(task):
    set_seed(SEED)
    labels = sorted(train_df[task].unique())
    label2id = {l: i for i, l in enumerate(labels)}
    id2label = {i: l for l, i in label2id.items()}
    tr, va, te = (to_ds(f, task, label2id) for f in (train_df, val_df, test_df))
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(labels), id2label=id2label, label2id=label2id)

    def metrics(p):
        pred = p.predictions.argmax(-1)
        return {"macro_f1": f1_score(p.label_ids, pred, average="macro"), "accuracy": accuracy_score(p.label_ids, pred)}

    args = TrainingArguments(
        output_dir=f"/content/runs/{task}", eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
        save_total_limit=1, learning_rate=HPARAMS["learning_rate"],
        per_device_train_batch_size=HPARAMS["batch_size"], per_device_eval_batch_size=64,
        num_train_epochs=HPARAMS["epochs"], warmup_steps=HPARAMS["warmup_fraction"],  # float = fraction in v5
        weight_decay=HPARAMS["weight_decay"], fp16=True, logging_steps=50, report_to="none", seed=SEED)
    weights = balanced_weights(train_df[task], labels)
    print(task, "class weights:", dict(zip(labels, [round(w, 2) for w in weights.tolist()])))
    trainer = WeightedTrainer(
        model=model, args=args, train_dataset=tr, eval_dataset=va, class_weights=weights,
        data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=HPARAMS["early_stopping_patience"])])
    t0 = time.time()
    trainer.train()
    train_s = time.time() - t0

    val_m = trainer.evaluate(va)
    out = trainer.predict(te)
    probs = torch.softmax(torch.tensor(out.predictions), -1).numpy()
    pred = probs.argmax(-1)
    y = np.array(out.label_ids)
    test_m = {
        "accuracy": round(accuracy_score(y, pred), 4),
        "macro_f1": round(f1_score(y, pred, average="macro"), 4),
        "weighted_f1": round(f1_score(y, pred, average="weighted"), 4),
        "per_class": classification_report(y, pred, labels=list(range(len(labels))), target_names=labels,
                                           output_dict=True, zero_division=0),
        "confusion_matrix": confusion_matrix(y, pred, labels=list(range(len(labels)))).tolist(),
    }
    task_dir = f"{OUT_DIR}/{task}"
    trainer.save_model(f"{task_dir}/model")
    tokenizer.save_pretrained(f"{task_dir}/model")
    test_df.assign(pred=[id2label[i] for i in pred], confidence=probs.max(-1))[
        ["ticket_id", task, "pred", "confidence"]].to_parquet(f"{task_dir}/test_predictions.parquet", index=False)
    report = {
        "run": RUN_NAME, "task": task, "model": MODEL_NAME, "max_len": MAX_LEN, "hparams": HPARAMS, "labels": labels,
        "fingerprints": got, "gpu": torch.cuda.get_device_name(0), "train_seconds": round(train_s, 1),
        "epochs_run": round(trainer.state.epoch, 2), "val_macro_f1": round(val_m["eval_macro_f1"], 4),
        "test": test_m, "log_history": trainer.state.log_history,
    }
    with open(f"{task_dir}/report.json", "w") as f:
        json.dump(report, f, indent=2)
    print(f"{task}: val macro-F1 {report['val_macro_f1']:.3f} | test macro-F1 {test_m['macro_f1']:.3f} "
          f"acc {test_m['accuracy']:.3f} | {train_s/60:.1f} min")
    return report


reports = {task: train_task(task) for task in TASKS}

token length p50/p95/max: 95 126 222


Map:   0%|          | 0/18923 [00:00<?, ? examples/s]

Map:   0%|          | 0/2365 [00:00<?, ? examples/s]

Map:   0%|          | 0/2366 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Macro F1,Accuracy
0.032091,1.613411,10,0.709251,0.717970


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

priority: val macro-F1 0.709 | test macro-F1 0.716 acc 0.720 | 16.2 min


## 6. Summary

In [7]:
for task, r in reports.items():
    print(f"{task:9s} test macro-F1 {r['test']['macro_f1']:.3f}  accuracy {r['test']['accuracy']:.3f}  "
          f"epochs {r['epochs_run']}  train {r['train_seconds']/60:.1f} min")
print("\nSaved to:", OUT_DIR)
print("Next: download that folder (queue/ and priority/) into the project as models/distilbert/, then run")
print("  python scripts/import_colab_results.py")

queue     test macro-F1 0.652  accuracy 0.621  epochs 10.0  train 15.8 min
priority  test macro-F1 0.716  accuracy 0.720  epochs 10.0  train 16.2 min

Saved to: /content/drive/MyDrive/SupportPilot/ticket_classifier/run2
Next: download that folder (queue/ and priority/) into the project as models/distilbert/, then run
  python scripts/import_colab_results.py
